# 2. Reward Model — 선호 학습

> 책 Chapter 6 — *Reward Modeling*

SFT 후 두 번째 단계. **인간 선호 데이터로부터 응답에 점수를 매기는 함수 $r_\phi(x, y)$를 학습**한다. 이 reward는 다음 단계(PPO)에서 RL 신호로 쓰인다.

## 핵심 질문

- **왜 인간이 직접 reward를 매기지 않고 모델을 학습시키나?** 매 PPO step마다 인간을 부를 순 없으니까. RM은 인간 선호의 *amortized approximation*.
- **선호 데이터에서 어떻게 scalar reward를 학습하나?** Bradley-Terry 모델 + sigmoid cross-entropy.
- **왜 그게 잘 되나?** Random utility theory의 통계적 정당성. 아래에서 유도.

## 1. 데이터 형식 — Preference triples

데이터셋: $\mathcal{D}_\mathrm{pref} = \{(x^{(i)}, y_c^{(i)}, y_r^{(i)})\}$

- $x$: prompt
- $y_c$: **c**hosen (인간이 더 선호한 응답)
- $y_r$: **r**ejected

UltraFeedback 같은 데이터셋이 이 형식. 보통 같은 prompt에 대한 2개 응답을 사람(또는 GPT-4)이 비교.

## 2. Bradley-Terry 모델

두 옵션 A, B 중 인간이 어느 쪽을 선호할 확률을 모델링한 통계 모형 (Bradley & Terry, 1952).

각 옵션에 잠재 "효용" $r$이 있고:

$$
P(A \succ B) = \frac{e^{r_A}}{e^{r_A} + e^{r_B}} = \sigma(r_A - r_B)
$$

여기서 $\sigma(z) = 1 / (1 + e^{-z})$는 sigmoid.

### 직관

두 응답의 reward 차이가 클수록 선호 확률이 1에 가까워지고, 같으면 0.5. 자연스럽다.

### LLM에 적용

$r$을 학습하는 신경망 $r_\phi(x, y)$로 두면:

$$
P(y_c \succ y_r \mid x) = \sigma\bigl(r_\phi(x, y_c) - r_\phi(x, y_r)\bigr)
$$

## 3. Loss 유도 — Negative Log Likelihood

데이터셋에서 인간이 항상 $y_c$를 선택했으므로 (선호도가 1):

$$
\mathcal{L}_\mathrm{RM}(\phi) = - \mathbb{E}_{(x, y_c, y_r) \sim \mathcal{D}_\mathrm{pref}}
\Bigl[ \log P(y_c \succ y_r \mid x) \Bigr]
$$

대입:

$$
\boxed{\mathcal{L}_\mathrm{RM}(\phi) = - \mathbb{E}\bigl[\log \sigma\bigl(r_\phi(x, y_c) - r_\phi(x, y_r)\bigr)\bigr]}
$$

이게 RM의 핵심 loss다. **Binary cross-entropy with logit = $r_c - r_r$, target = 1**과 동등.

### 다른 형태로 쓰면

$\log \sigma(z) = -\log(1 + e^{-z})$ 라서:

$$
\mathcal{L}_\mathrm{RM} = \mathbb{E}\bigl[\log(1 + e^{-(r_c - r_r)})\bigr]
$$

즉 **softplus** of the negative reward margin.

### 왜 이게 잘 동작하나?

이 loss를 미니마이즈하면:
- $r_c > r_r$이 되도록 학습 (margin 양수화)
- 모든 pair에 대해 일관된 ranking을 유도

수학적으로: Bradley-Terry 모델 자체가 *random utility model*의 한 형태이고, 충분한 데이터가 있으면 효용 함수가 affine equivalence까지 식별 가능함이 알려져 있음.

## 4. Reward Model 아키텍처

기본 구조:
1. **Backbone**: SFT 모델 (Kybalion-1B)
2. **Head**: 마지막 hidden state(또는 EOS 위치) → scalar로 매핑하는 linear layer

```
[input_ids] → Backbone Transformer → hidden states (T, D) → take last token's hidden → Linear(D, 1) → scalar reward
```

LM head는 제거하고 새 scalar head 학습. backbone은 둘 중 선택:
- **Fully fine-tune** (Zephyr, InstructGPT 표준)
- **Freeze + LoRA**: 더 빠르고 메모리 적음, 약간 약함

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional


class RewardModel(nn.Module):
    """Reward Model: SFT backbone + scalar value head.

    응답 시퀀스의 마지막 non-pad 토큰의 hidden state를 사용해 scalar reward를 출력.

    Args:
        backbone: HF causal LM (예: AutoModelForCausalLM.from_pretrained("devwoo/Kybalion-1B"))
        hidden_size: backbone의 hidden dimension (Kybalion-1B는 2048)
    """

    def __init__(self, backbone, hidden_size: int):
        super().__init__()
        self.backbone = backbone
        # Scalar head — bias 없음 (선호의 절대 zero point는 의미 없으니)
        self.v_head = nn.Linear(hidden_size, 1, bias=False)
        nn.init.normal_(self.v_head.weight, std=1.0 / (hidden_size + 1))

    def forward(
        self,
        input_ids: torch.Tensor,           # (B, T)
        attention_mask: torch.Tensor,      # (B, T) — pad=0, valid=1
    ) -> torch.Tensor:                     # (B,) — 시퀀스당 scalar reward
        # 1) Backbone forward, hidden state 받기
        #    HF model의 output_hidden_states=True로 마지막 hidden 가져옴
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            use_cache=False,
        )
        hidden_states = outputs.hidden_states[-1]      # (B, T, D)

        # 2) 시퀀스마다 마지막 non-pad 위치의 hidden state 추출
        #    attention_mask로 valid length 계산
        seq_lengths = attention_mask.sum(dim=-1) - 1   # (B,) — 마지막 유효 토큰 인덱스
        batch_idx = torch.arange(input_ids.size(0), device=input_ids.device)
        last_hidden = hidden_states[batch_idx, seq_lengths]   # (B, D)

        # 3) Scalar head — (B, 1) → (B,)
        reward = self.v_head(last_hidden).squeeze(-1)
        return reward


# 시연 — 실제 backbone 없이 dummy로 흐름만 확인
"""
from transformers import AutoModelForCausalLM
backbone = AutoModelForCausalLM.from_pretrained("devwoo/Kybalion-1B")
rm = RewardModel(backbone, hidden_size=backbone.config.hidden_size)
"""
print("RewardModel class 정의 완료")

## 5. Bradley-Terry Loss를 직접 구현

In [ ]:
def bradley_terry_loss(
    reward_chosen: torch.Tensor,    # (B,)
    reward_rejected: torch.Tensor,  # (B,)
) -> dict[str, torch.Tensor]:
    """RM의 핵심 loss: -log σ(r_c - r_r)

    Args:
        reward_chosen:   chosen 응답에 대한 RM 점수
        reward_rejected: rejected 응답에 대한 RM 점수

    Returns:
        dict with:
            loss     — 평균 loss (스칼라)
            accuracy — chosen > rejected 비율 (학습 모니터링)
            margin   — r_chosen - r_rejected 평균
    """
    # margin = r_c - r_r. 양수이면 chosen이 더 좋다고 판단.
    margin = reward_chosen - reward_rejected   # (B,)

    # softplus(-x) = log(1 + e^{-x}) = -log σ(x).
    # F.logsigmoid를 직접 써서 수치적으로 안정한 구현.
    loss = -F.logsigmoid(margin).mean()

    # 학습 모니터링용 지표
    accuracy = (margin > 0).float().mean()
    return {
        "loss": loss,
        "accuracy": accuracy,
        "margin": margin.mean(),
        "reward_chosen_mean": reward_chosen.mean(),
        "reward_rejected_mean": reward_rejected.mean(),
    }


# 시연
torch.manual_seed(0)
r_c = torch.randn(8) + 0.5   # chosen이 평균적으로 +0.5
r_r = torch.randn(8)
info = bradley_terry_loss(r_c, r_r)
for k, v in info.items():
    print(f"{k:25s} {v.item():.4f}")

## 6. Loss 유도 검증 — 수동 계산과 일치하는지

위 한 줄 짜리 loss가 정말 BT 모델의 NLL인지 sanity check.

In [ ]:
def bradley_terry_loss_manual(r_c: torch.Tensor, r_r: torch.Tensor) -> torch.Tensor:
    """L = - log σ(r_c - r_r) 를 단계별로 풀어 작성."""
    margin = r_c - r_r
    p = torch.sigmoid(margin)              # P(chosen > rejected)
    nll = -torch.log(p + 1e-12)            # numerical safety
    return nll.mean()


torch.manual_seed(0)
r_c = torch.randn(16)
r_r = torch.randn(16)

a = bradley_terry_loss(r_c, r_r)["loss"]
b = bradley_terry_loss_manual(r_c, r_r)
print(f"F.logsigmoid 사용: {a.item():.6f}")
print(f"수동 σ + log    : {b.item():.6f}")
print(f"동일?            {torch.allclose(a, b, atol=1e-5)}")

## 7. Dataset & DataLoader — pairwise

각 example이 두 시퀀스(chosen, rejected)를 담는 구조. 보통 batch당 forward를 두 번 (또는 concatenate해서 한 번).

In [ ]:
from torch.utils.data import Dataset


class PreferenceDataset(Dataset):
    """Preference pair dataset.

    UltraFeedback 같은 데이터셋을 (chosen_input_ids, rejected_input_ids) 형태로 변환.
    """

    def __init__(self, raw_examples: list[dict], tokenizer, max_length: int = 1024):
        self.examples = raw_examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def _encode(self, prompt: str, response: str) -> torch.Tensor:
        """프롬프트 + 응답을 chat template로 묶어 토큰화."""
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
        ids = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=False,
            return_tensors="pt",
        ).squeeze(0)
        return ids[: self.max_length]

    def __getitem__(self, idx):
        ex = self.examples[idx]
        chosen_ids   = self._encode(ex["prompt"], ex["chosen"])
        rejected_ids = self._encode(ex["prompt"], ex["rejected"])
        return {"chosen_ids": chosen_ids, "rejected_ids": rejected_ids}


def collate_preferences(batch: list[dict], pad_token_id: int) -> dict[str, torch.Tensor]:
    """Pad both chosen and rejected to the global max length in the batch.

    선택 사항: chosen과 rejected를 합쳐서 한 번에 forward (메모리 효율).
    """
    all_ids = [b["chosen_ids"] for b in batch] + [b["rejected_ids"] for b in batch]
    max_len = max(x.size(0) for x in all_ids)
    B = len(batch)

    padded = torch.full((2 * B, max_len), pad_token_id, dtype=torch.long)
    attn = torch.zeros(2 * B, max_len, dtype=torch.long)
    for i, ids in enumerate(all_ids):
        L = ids.size(0)
        padded[i, :L] = ids
        attn[i, :L] = 1

    # 첫 B개는 chosen, 뒤 B개는 rejected
    return {
        "input_ids": padded,             # (2B, T)
        "attention_mask": attn,          # (2B, T)
        "B": B,
    }

## 8. 학습 step — Concatenated forward 트릭

chosen과 rejected를 *별도로* forward하면 backbone을 두 번 돌려야 함. 더 효율적으로: 하나의 (2B, T) 텐서로 concat해서 한 번에 forward, 결과 reward를 둘로 split.

In [ ]:
def rm_train_step(rm: RewardModel, batch: dict, optimizer) -> dict[str, float]:
    """Reward Model 학습 한 step.

    chosen + rejected를 (2B, T) 한 배치로 묶어 한 번만 forward → reward 분리 → BT loss.
    """
    B = batch["B"]

    # 1) Concat된 (2B, T)로 한 번에 forward
    rewards = rm(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
    )                                            # (2B,)

    # 2) 앞 B개는 chosen, 뒤 B개는 rejected
    r_c = rewards[:B]                            # (B,)
    r_r = rewards[B:]                            # (B,)

    # 3) Bradley-Terry loss
    info = bradley_terry_loss(r_c, r_r)
    loss = info["loss"]

    # 4) Backward + step
    loss.backward()
    torch.nn.utils.clip_grad_norm_(rm.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()

    # 학습 모니터링
    return {k: v.item() for k, v in info.items()}


# 학습 loop 의사 코드:
"""
for epoch in range(num_epochs):
    for batch in dataloader:
        stats = rm_train_step(rm, batch, optimizer)
        if step % 100 == 0:
            print(f"step {step} | loss {stats['loss']:.4f} | acc {stats['accuracy']:.3f} | margin {stats['margin']:.3f}")
"""
print("RM 학습 step 구현 완료")

## 9. Reward Model의 실패 모드 — Reward Hacking

RM을 충분히 학습시킨 후 PPO에서 reward로 사용할 때, 정책이 reward를 *exploit*할 위험.

### 예시

- RM이 "긴 응답"에 높은 점수를 주는 경향을 학습했다면 → PPO 정책은 *무의미하게 긴 응답*을 생성
- RM이 특정 키워드("certainly", "I'd be happy to")에 보너스를 준다면 → 정책은 그 키워드를 *남발*

### 대응

1. **KL penalty**: PPO에서 SFT로부터 너무 멀어지지 못하게 (다음 노트북에서 다룸)
2. **RM regularization**: BT loss에 L2 정규화 추가
3. **Diverse data**: chosen과 rejected가 stylistic feature가 아니라 *content*로 구분되도록 데이터 큐레이션
4. **Reward shaping**: 학습 후 reward를 z-normalize

## 10. PPO를 위한 정리

RM 학습이 끝나면:
- **고정 모델**: $r_\phi$ — PPO 학습 중에 freeze. gradient 흐르지 않음.
- **출력 형식**: 시퀀스 단위 scalar. 인간이 "전체 응답"을 보고 평가한 신호.

→ PPO 노트북에서는 **응답 끝에서 한 번의 reward 신호**를 받아 RL의 보상으로 사용하게 된다.

## 11. 다음 노트북 — `3_PPO.ipynb`

마지막이자 가장 큰 노트북. **Policy Gradient → REINFORCE → Advantage → GAE → PPO clipped objective → KL penalty → 풀 RLHF 루프** 순서로 빌드업한다.